# 실습: OpenAI 제어 파라미터

같은 프롬프트라도 파라미터를 바꾸면 길이, 다양성, 추론 깊이, 중단 위치가 달라집니다.


---
## 실습 전 주의

- 모든 모델이 모든 파라미터를 지원하지는 않습니다.
- `BadRequestError`가 나오면 모델과 파라미터 조합을 확인합니다.
- 비교 실습은 비용을 줄이기 위해 `max_output_tokens`를 작게 둡니다.
---


In [1]:
import os

from dotenv import load_dotenv
from openai import APIError, BadRequestError, OpenAI, RateLimitError

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError(".env 파일에 OPENAI_API_KEY를 먼저 설정하세요.")

MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
client = OpenAI(api_key=api_key)

print(f"사용 모델: {MODEL}")


사용 모델: gpt-5.4-mini


In [2]:
def show_usage(response) -> None:
    usage = getattr(response, "usage", None)
    if usage:
        print(
            f"\n토큰: input={usage.input_tokens}, "
            f"output={usage.output_tokens}, total={usage.total_tokens}"
        )


def generate(label: str, prompt: str, **params):
    print(f"\n--- {label} ---")
    try:
        response = client.responses.create(
            model=MODEL,
            input=prompt,
            **params,
        )
    except BadRequestError as exc:
        print("BadRequestError:", getattr(exc, "message", str(exc)))
        return None
    except RateLimitError as exc:
        print("RateLimitError:", exc)
        return None
    except APIError as exc:
        print("APIError:", exc)
        return None
    except TypeError as exc:
        print("TypeError: 지원하지 않는 SDK 파라미터입니다.", exc)
        return None

    print(response.output_text)
    show_usage(response)
    return response


---
## 1. 기준 프롬프트


In [3]:
base_prompt = (
    "초보 개발자가 함수형 프로그래밍을 이해할 수 있도록 "
    "핵심 개념과 Python 예시를 설명해줘."
)

baseline = generate(
    "기준 응답",
    base_prompt,
    max_output_tokens=250,
)



--- 기준 응답 ---
물론입니다.  
**함수형 프로그래밍(Function Programming, FP)**은 “프로그램을 함수들의 조합으로 구성하는 방식”입니다.  
초보 개발자 입장에서 이해하기 쉽게 말하면, **데이터를 직접 바꾸기보다, 입력을 받아 새 결과를 만들어내는 함수들을 중심으로 코드를 짜는 방식**입니다.

---

# 1. 함수형 프로그래밍의 핵심 아이디어

## 1) 함수는 “입력 → 출력”에 집중한다
함수형 프로그래밍에서는 함수가 가능한 한:

- 같은 입력이면 항상 같은 출력
- 외부 상태를 몰라도 동작
- 내부에서 값을 바꾸지 않음

이런 방식으로 작성됩니다.

### 예시
```python
def add(a, b):
    return a + b
```

`add(2, 3)`은 언제 호출해도 `5`를 반환합니다.  
이런 함수가 함수형 프로그래밍에서 중요합니다.

---

## 2) 불변성(Immutability)
데이터를 **직접 수정하지 않고**, 새 데이터를 만들어 사용합니다.

### 일반적인 방식
```python

토큰: input=33, output=250, total=283


---
## 2. `max_output_tokens`: 출력 길이 제한


In [4]:
for limit in [60, 160, 320]:
    generate(
        f"max_output_tokens={limit}",
        base_prompt,
        max_output_tokens=limit,
    )



--- max_output_tokens=60 ---
물론입니다. **함수형 프로그래밍(Functional Programming, FP)**은 “프로그램을 상태를 바꾸는 명령들의 나열”로 보기보다, **함수를 조합해서 문제를 해결하는 방식**으로 생각하는 패러다임입니다

토큰: input=33, output=60, total=93

--- max_output_tokens=160 ---
물론입니다. 초보 개발자 기준으로 **함수형 프로그래밍(Functional Programming, FP)** 을 이해하기 쉽게 핵심 개념과 Python 예시로 설명해볼게요.

---

# 1. 함수형 프로그래밍이란?

함수형 프로그래밍은 프로그램을 **“값을 바꾸는 명령들의 모음”** 이 아니라,  
**“함수들을 조합해서 문제를 해결하는 방식”** 으로 보는 스타일입니다.

즉, 핵심은:

- **데이터를 직접 바꾸기보다**
- **입력을 받아 결과를 돌려주는 함수**를 중심으로
- **작은 함수들을 조합**해서 프로그램을 만든다

---

# 2

토큰: input=33, output=160, total=193

--- max_output_tokens=320 ---
물론이죠.  
초보 개발자 관점에서 **함수형 프로그래밍(Functional Programming, FP)** 을 이해하기 쉽게 설명하고, **Python 예시**로 같이 보겠습니다.

---

## 1. 함수형 프로그래밍이란?

함수형 프로그래밍은 **“데이터를 바꾸기보다, 함수를 조합해서 문제를 해결하는 방식”** 입니다.

핵심은:

- **함수를 값처럼 다루고**
- **상태를 최대한 바꾸지 않으며**
- **같은 입력이면 항상 같은 출력을 내는 함수**를 선호하는 것

즉, “어떻게 상태를 바꿀까?”보다  
“입력을 받아 결과를 계산하는 함수를 어떻게 잘 조합할까?”에 집중합니다.

---

## 2. 함수형 프로그래밍의 핵심 개념

### 2-1. 순수 함수(Pure Function)

순수 함수는 다음

---
## 3. `temperature`: 무작위성 조절

낮을수록 안정적이고 비슷한 답변이 나오기 쉽습니다.
높을수록 표현이 다양해질 수 있습니다.

모델에 따라 `temperature` 변경을 지원하지 않을 수 있습니다.


In [5]:
creative_prompt = "데이터 분석가를 위한 Python 학습 슬로건을 3개 만들어줘."

for temperature in [0.2, 0.8, 1.4]:
    generate(
        f"temperature={temperature}",
        creative_prompt,
        temperature=temperature,
        max_output_tokens=160,
    )



--- temperature=0.2 ---
물론입니다. 데이터 분석가를 위한 Python 학습 슬로건 3개입니다:

1. **Python으로 데이터의 숨은 이야기를 읽어내자**
2. **분석의 속도를 높이는 힘, Python**
3. **데이터를 이해하는 가장 스마트한 방법, Python**

원하시면 **더 캐치한 스타일**, **짧고 강한 스타일**, 또는 **회사/교육용 톤**으로도 다시 만들어드릴게요.

토큰: input=25, output=105, total=130

--- temperature=0.8 ---
물론입니다. 데이터 분석가를 위한 Python 학습 슬로건 3개입니다:

1. **“Python으로 데이터의 숨은 이야기를 읽어내자.”**  
2. **“분석은 더 깊게, Python은 더 스마트하게.”**  
3. **“데이터를 이해하는 가장 강력한 언어, Python.”**  

원하시면 **더 캐치한 스타일**, **짧은 캠페인 문구 스타일**, 또는 **회사 교육용 슬로건**으로도 바꿔드릴게요.

토큰: input=25, output=123, total=148

--- temperature=1.4 ---
물론이죠. 데이터 분석가를 위한 Python 학습 슬로건 3개를 제안할게요.

1. **“Python으로 데이터는 읽고, 인사이트는 만든다.”**  
2. **“분석의 속도를 높이는 가장 강력한 도구, Python.”**  
3. **“데이터 분석가의 경쟁력은 Python에서 시작된다.”**  

원하시면 제가 이 슬로건들을 **더 캐치하게**, **더 전문적으로**, 또는 **광고 문구처럼 짧게** 다시 다듬어드릴게요.

토큰: input=25, output=131, total=156


---
## 4. `top_p`: 후보 토큰 범위 조절

`top_p`는 누적 확률 기준으로 후보 토큰의 범위를 제한합니다.
보통 `temperature`와 동시에 크게 바꾸기보다 하나씩 비교합니다.


In [6]:
for top_p in [0.4, 0.8, 1.0]:
    generate(
        f"top_p={top_p}",
        creative_prompt,
        top_p=top_p,
        max_output_tokens=160,
    )



--- top_p=0.4 ---
물론입니다. 데이터 분석가를 위한 Python 학습 슬로건 3개입니다:

1. **Python으로 데이터의 숨은 이야기를 읽어내자**
2. **분석은 더 빠르게, 인사이트는 더 깊게 — Python과 함께**
3. **데이터를 코드로, 통찰을 결과로 바꾸는 힘, Python**

원하시면 **더 캐주얼한 버전**, **더 전문적인 버전**, 또는 **짧고 강한 한 줄 슬로건**으로도 다시 만들어드릴게요.

토큰: input=25, output=123, total=148

--- top_p=0.8 ---
물론입니다. 데이터 분석가를 위한 Python 학습 슬로건 3개입니다:

1. **Python으로 데이터의 숨은 이야기를 읽자**
2. **분석의 시작은 Python, 인사이트의 완성도 Python**
3. **데이터를 이해하는 가장 빠른 길, Python**

원하시면 **더 캐치한 느낌**, **공식적인 느낌**, 또는 **슬로건형으로 짧고 강하게** 다시 만들어드릴게요.

토큰: input=25, output=107, total=132

--- top_p=1.0 ---
물론입니다. 데이터 분석가를 위한 Python 학습 슬로건 3개입니다:

1. **“Python으로 데이터를 읽고, 인사이트로 답하라.”**  
2. **“분석의 속도는 Python, 결정의 힘은 데이터.”**  
3. **“코드 한 줄로 데이터의 가치를 깨워라.”**  

원하시면 **더 짧고 강렬한 버전**, **기업/교육용 톤**, 또는 **영문 슬로건**으로도 바꿔드릴게요.

토큰: input=25, output=123, total=148


---
## 5. Responses API에서 `stop` 대신 구분자 후처리

`client.responses.create(...)`는 `stop` 파라미터를 받지 않습니다.
특정 구분자에서 화면에 보여줄 내용을 끊어야 한다면 응답을 받은 뒤 문자열로 나누어 처리합니다.


In [7]:
stop_prompt = (
    "다음 형식으로 세 단계를 작성해줘. "
    "각 단계 끝에는 ---끝---을 붙여줘. "
    "주제: API 키를 안전하게 관리하는 법"
)

generate(
    "stop 없이 실행",
    stop_prompt,
    max_output_tokens=250,
)

generate(
    "stop=['---끝---'] 적용",
    stop_prompt,
    stop=["---끝---"],
    max_output_tokens=250,
)



--- stop 없이 실행 ---
1단계: API 키는 코드에 직접 넣지 말고 환경 변수나 비밀 관리 도구에 저장한다.  
예를 들어 `.env` 파일, AWS Secrets Manager, Google Secret Manager, HashiCorp Vault 등을 사용하면 코드와 키를 분리할 수 있다.  
---끝---

2단계: API 키 접근 권한을 최소화하고 주기적으로 교체한다.  
필요한 서비스와 사람만 접근하도록 권한을 제한하고, 키가 유출됐을 때를 대비해 정기적으로 로테이션하는 것이 좋다.  
---끝---

3단계: 키 사용 내역을 모니터링하고, 외부 저장소에 노출되지 않도록 주의한다.  
로그나 Git 저장소에 키가 남지 않게 하고, 이상 사용이 발견되면 즉시 폐기 및 재발급한다.  
---끝---

토큰: input=41, output=193, total=234

--- stop=['---끝---'] 적용 ---
TypeError: 지원하지 않는 SDK 파라미터입니다. Responses.create() got an unexpected keyword argument 'stop'


---
## 6. `reasoning`: 추론 노력 조절

GPT-5 계열과 o 계열 reasoning 모델은 `reasoning` 설정을 사용할 수 있습니다.
값을 높이면 더 깊게 생각할 수 있지만 응답 시간이 늘거나 토큰 사용량이 증가할 수 있습니다.


In [8]:
reasoning_prompt = (
    "한 수업에서 OpenAI, Gemini, Ollama를 어떤 순서로 가르치면 좋을지 "
    "이유와 함께 3단계로 제안해줘."
)

for effort in ["low", "medium", "high"]:
    generate(
        f"reasoning effort={effort}",
        reasoning_prompt,
        reasoning={"effort": effort},
        max_output_tokens=260,
    )



--- reasoning effort=low ---
수업에서는 보통 **OpenAI → Gemini → Ollama** 순서로 가르치는 것이 가장 자연스럽습니다.  
이유는 **개념 난이도, 실습 환경의 안정성, 그리고 확장성**을 단계적으로 익히기 좋기 때문입니다.

## 1단계: OpenAI — LLM/API 기본기 익히기
**왜 먼저?**  
- 가장 널리 알려진 형태의 **클라우드 API 기반 LLM 사용법**을 배우기 좋습니다.
- 챗봇, 함수 호출, 프롬프트 설계, 스트리밍 응답 등 **기본 개념을 표준처럼** 익힐 수 있습니다.
- 문서와 예제가 많아 초보자도 따라가기 쉽습니다.

**이 단계에서 배우면 좋은 것**
- API 요청/응답 구조
-

토큰: input=42, output=260, total=302

--- reasoning effort=medium ---
수업에서는 **OpenAI → Gemini → Ollama** 순서로 가르치는 것을 추천합니다.  
이 순서가 좋은 이유는 **기본 개념 → 확

토큰: input=42, output=260, total=302

--- reasoning effort=high ---


토큰: input=42, output=260, total=302


---
## 7. `instructions`와 `input` 역할 분리


In [9]:
prompt = "클래스와 객체의 차이를 알려줘."

generate(
    "일반 설명",
    prompt,
    max_output_tokens=220,
)

generate(
    "면접 대비 설명",
    prompt,
    instructions=(
        "당신은 주니어 개발자 면접 코치입니다. "
        "정의, 비유, 면접 답변 예시 순서로 작성하세요."
    ),
    max_output_tokens=320,
)



--- 일반 설명 ---
물론이야. 아주 간단히 말하면:

- **클래스(Class)**: 객체를 만들기 위한 **설계도**
- **객체(Object)**: 그 설계도로 실제 만든 **실체**

## 예시
예를 들어 `Car`라는 클래스가 있다고 해보자.

- 클래스 `Car`에는  
  - 색깔
  - 브랜드
  - 속도  
  같은 속성과
  - 달리기
  - 멈추기  
  같은 기능이 정의될 수 있어.

이 클래스를 바탕으로 실제로 만든 자동차 한 대 한 대가 **객체**야.

예:
- `내 차 = 빨간색 현대차`
- `친구 차 = 파란색 기아차`

이 둘은 같은 `Car` 클래스에서 만들어졌지만, 값은 각각 다를 수 있어.

## 차이 정리

| 구분 | 클래스 | 객체 |
|------|--------|------|
| 의미 | 설계도, 틀 | 설계도로 만든 실

토큰: input=17, output=220, total=237

--- 면접 대비 설명 ---
### 1) 정의
- **클래스(Class)**: 객체를 만들기 위한 **설계도**입니다.  
  속성(데이터)과 기능(메서드)을 어떤 형태로 가질지 정의합니다.
- **객체(Object)**: 그 설계도를 바탕으로 실제로 만들어진 **실체**입니다.  
  클래스에 정의된 내용을 가지고 메모리상에 생성된 인스턴스입니다.

### 2) 비유
- **클래스 = 붕어빵 틀**
- **객체 = 붕어빵**

틀 하나로 여러 개의 붕어빵을 만들 수 있듯이, 하나의 클래스로 여러 객체를 만들 수 있습니다.  
즉, 클래스는 공통 구조를 정하고, 객체는 그 구조를 가진 실제 존재입니다.

### 3) 면접 답변 예시
“클래스는 객체를 만들기 위한 설계도이고, 객체는 그 설계도로 생성된 실제 인스턴스입니다. 클래스에는 속성과 기능을 정의하고, 객체는 그 정의를 바탕으로 메모리에 생성되어 실제로 사용됩니다. 예를 들어 `Car`라는 클래스가 있으면, `myCar`, `yourCar` 같은 여러 객체를 만들 수 

Response(id='resp_0838d0158e9e1bac006a4726410d488198bec043c1ddf0437c', created_at=1783047745.0, error=None, incomplete_details=None, instructions='당신은 주니어 개발자 면접 코치입니다. 정의, 비유, 면접 답변 예시 순서로 작성하세요.', metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_0838d0158e9e1bac006a47264161d48198b4e4fe6070df8179', content=[ResponseOutputText(annotations=[], text='### 1) 정의\n- **클래스(Class)**: 객체를 만들기 위한 **설계도**입니다.  \n  속성(데이터)과 기능(메서드)을 어떤 형태로 가질지 정의합니다.\n- **객체(Object)**: 그 설계도를 바탕으로 실제로 만들어진 **실체**입니다.  \n  클래스에 정의된 내용을 가지고 메모리상에 생성된 인스턴스입니다.\n\n### 2) 비유\n- **클래스 = 붕어빵 틀**\n- **객체 = 붕어빵**\n\n틀 하나로 여러 개의 붕어빵을 만들 수 있듯이, 하나의 클래스로 여러 객체를 만들 수 있습니다.  \n즉, 클래스는 공통 구조를 정하고, 객체는 그 구조를 가진 실제 존재입니다.\n\n### 3) 면접 답변 예시\n“클래스는 객체를 만들기 위한 설계도이고, 객체는 그 설계도로 생성된 실제 인스턴스입니다. 클래스에는 속성과 기능을 정의하고, 객체는 그 정의를 바탕으로 메모리에 생성되어 실제로 사용됩니다. 예를 들어 `Car`라는 클래스가 있으면, `myCar`, `yourCar` 같은 여러 객체를 만들 수 있습니다.”\n\n원하시면 이어서 **“객체지향에서 인스턴스와 객체의 차이”**도 같이 설명해드릴게요.', type='output_t

---
## 8. 실무 레시피


In [10]:
# 레시피 1: 사실 기반 Q&A
generate(
    "사실 기반 Q&A",
    "Python 가상환경을 쓰는 이유를 4가지로 정리해줘.",
    instructions="불확실한 내용은 추측하지 말고, 초보자에게 필요한 내용만 답하세요.",
    reasoning={"effort": "low"},
    max_output_tokens=240,
)



--- 사실 기반 Q&A ---
Python 가상환경을 쓰는 이유는 보통 다음 4가지입니다.

1. **프로젝트마다 패키지 버전을 따로 관리하기 위해**
   - 어떤 프로젝트는 Django 3.x가 필요하고, 다른 프로젝트는 4.x가 필요할 수 있습니다.
   - 가상환경을 쓰면 서로 충돌하지 않게 분리할 수 있습니다.

2. **전역 Python 환경을 깨끗하게 유지하기 위해**
   - 시스템 전체에 패키지를 막 설치하면 나중에 어떤 패키지가 왜 설치됐는지 관리가 어려워집니다.
   - 가상환경을 사용하면 프로젝트에 필요한 것만 따로 설치할 수 있습니다.

3. **의존성 충돌을 줄이기 위해**
   - 같은 패키지라도 프로젝트마다 필요한 버전이 다를 수 있습니다.
   - 가상환경은 이런 충돌을 줄여서 실행 오류를 예방하는 데 도움이 됩니다.

4. **협업과 재현을 쉽게 하기 위해**
   - `requirements.txt` 같은 파일로 필요한

토큰: input=50, output=240, total=290


Response(id='resp_0aeaecc60bf7c2e5006a47264530708199ace577e78de6ec9a', created_at=1783047749.0, error=None, incomplete_details=IncompleteDetails(reason='max_output_tokens'), instructions='불확실한 내용은 추측하지 말고, 초보자에게 필요한 내용만 답하세요.', metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseReasoningItem(id='rs_0aeaecc60bf7c2e5006a4726456ae48199868b8a6c7008c93a', summary=[], type='reasoning', content=[], encrypted_content=None, status=None), ResponseOutputMessage(id='msg_0aeaecc60bf7c2e5006a4726457f1481999651637723b5139d', content=[ResponseOutputText(annotations=[], text='Python 가상환경을 쓰는 이유는 보통 다음 4가지입니다.\n\n1. **프로젝트마다 패키지 버전을 따로 관리하기 위해**\n   - 어떤 프로젝트는 Django 3.x가 필요하고, 다른 프로젝트는 4.x가 필요할 수 있습니다.\n   - 가상환경을 쓰면 서로 충돌하지 않게 분리할 수 있습니다.\n\n2. **전역 Python 환경을 깨끗하게 유지하기 위해**\n   - 시스템 전체에 패키지를 막 설치하면 나중에 어떤 패키지가 왜 설치됐는지 관리가 어려워집니다.\n   - 가상환경을 사용하면 프로젝트에 필요한 것만 따로 설치할 수 있습니다.\n\n3. **의존성 충돌을 줄이기 위해**\n   - 같은 패키지라도 프로젝트마다 필요한 버전이 다를 수 있습니다.\n   - 가상환경은 이런 충돌을 줄여서 실행 오류를 예

In [11]:
# 레시피 2: 자연스러운 대화
generate(
    "자연스러운 대화",
    "수업 중 API 오류가 났을 때 학생에게 어떻게 말하면 좋을까?",
    instructions="따뜻하고 짧은 말투로 답하세요.",
    max_output_tokens=180,
)



--- 자연스러운 대화 ---
이렇게 말하면 부드럽고 안정적으로 전달돼요:

- “지금 API 오류가 잠깐 발생했어요. 바로 확인해볼게요.”
- “걱정하지 마세요. 문제를 찾는 중입니다.”
- “일시적인 오류일 수 있어서 잠시만 기다려 주세요.”
- “불편을 드려 죄송합니다. 곧 정상적으로 이어가겠습니다.”

조금 더 자연스럽게 말하면:

- “잠깐 API 쪽에 문제가 생겼어요. 금방 확인해볼게요.”
- “지금은 오류가 있어서 잠시 멈췄지만, 바로 다시 진행할게요.”

원하시면 제가  
**더 친근한 버전 / 더 공손한 버전 / 학생용 쉬운 말**로 나눠서도 적어드

토큰: input=41, output=180, total=221


Response(id='resp_008064a74b1176f6006a4726473d68819b9868e7615c6f66d3', created_at=1783047751.0, error=None, incomplete_details=IncompleteDetails(reason='max_output_tokens'), instructions='따뜻하고 짧은 말투로 답하세요.', metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_008064a74b1176f6006a4726478b14819bb8d9965eb829a1b8', content=[ResponseOutputText(annotations=[], text='이렇게 말하면 부드럽고 안정적으로 전달돼요:\n\n- “지금 API 오류가 잠깐 발생했어요. 바로 확인해볼게요.”\n- “걱정하지 마세요. 문제를 찾는 중입니다.”\n- “일시적인 오류일 수 있어서 잠시만 기다려 주세요.”\n- “불편을 드려 죄송합니다. 곧 정상적으로 이어가겠습니다.”\n\n조금 더 자연스럽게 말하면:\n\n- “잠깐 API 쪽에 문제가 생겼어요. 금방 확인해볼게요.”\n- “지금은 오류가 있어서 잠시 멈췄지만, 바로 다시 진행할게요.”\n\n원하시면 제가  \n**더 친근한 버전 / 더 공손한 버전 / 학생용 쉬운 말**로 나눠서도 적어드', type='output_text', logprobs=[])], role='assistant', status='incomplete', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=0.98, background=False, completed_at=None, conversation=None, max_output_to

In [12]:
# 레시피 3: 창의적 글쓰기
generate(
    "창의적 글쓰기",
    "LLM API 수업을 홍보하는 짧은 문구 5개를 만들어줘.",
    temperature=1.1,
    top_p=0.95,
    max_output_tokens=220,
)



--- 창의적 글쓰기 ---
물론입니다. 아래처럼 짧고 홍보용으로 쓰기 좋은 문구 5개를 드릴게요.

1. **LLM API, 이제 직접 써보며 배워보세요!**  
2. **실무에 바로 쓰는 LLM API 수업, 지금 시작하세요.**  
3. **AI 서비스를 만드는 첫걸음, LLM API로 시작!**  
4. **쉽고 빠르게 배우는 LLM API 활용법.**  
5. **LLM API를 이해하면, AI 개발이 달라집니다.**

원하시면 제가 이걸  
- **더 캐주얼하게**,  
- **더 전문적으로**,  
- **광고 카피처럼 강하게**  
바꿔드릴 수도 있어요.

토큰: input=27, output=170, total=197


Response(id='resp_0657efe13270e380006a472649417c8198991a6f3ab1a114f8', created_at=1783047753.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_0657efe13270e380006a47264983748198ac6d77a5a89fddd7', content=[ResponseOutputText(annotations=[], text='물론입니다. 아래처럼 짧고 홍보용으로 쓰기 좋은 문구 5개를 드릴게요.\n\n1. **LLM API, 이제 직접 써보며 배워보세요!**  \n2. **실무에 바로 쓰는 LLM API 수업, 지금 시작하세요.**  \n3. **AI 서비스를 만드는 첫걸음, LLM API로 시작!**  \n4. **쉽고 빠르게 배우는 LLM API 활용법.**  \n5. **LLM API를 이해하면, AI 개발이 달라집니다.**\n\n원하시면 제가 이걸  \n- **더 캐주얼하게**,  \n- **더 전문적으로**,  \n- **광고 카피처럼 강하게**  \n바꿔드릴 수도 있어요.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.1, tool_choice='auto', tools=[], top_p=0.95, background=False, completed_at=1783047754.0, conversation=None, max_output_tokens=220, max_tool_calls=None, moderation=None,

In [13]:
# 레시피 4: 코드 생성
generate(
    "코드 생성",
    "python-dotenv로 .env의 OPENAI_API_KEY를 읽는 최소 예제를 작성해줘.",
    instructions="코드는 하나의 블록으로만 작성하고, 코드 아래에 주의점 2개를 붙이세요.",
    reasoning={"effort": "medium"},
    max_output_tokens=320,
)



--- 코드 생성 ---
```python
from dotenv import load_dotenv
import os

load_dotenv()  # 현재 디렉터리의 .env를 읽음

api_key = os.getenv("OPENAI_API_KEY")
print(api_key)
```

주의점 1. `.env` 파일에는 `OPENAI_API_KEY=your_key_here` 형식으로 넣고, 이 파일이 실행 중인 작업 디렉터리에서

토큰: input=56, output=320, total=376


Response(id='resp_0fb8e980b02340b8006a47264b5794819aa050c37b8f048cc1', created_at=1783047755.0, error=None, incomplete_details=IncompleteDetails(reason='max_output_tokens'), instructions='코드는 하나의 블록으로만 작성하고, 코드 아래에 주의점 2개를 붙이세요.', metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseReasoningItem(id='rs_0fb8e980b02340b8006a47264b99cc819aa356baf501ef88f7', summary=[], type='reasoning', content=[], encrypted_content=None, status=None), ResponseOutputMessage(id='msg_0fb8e980b02340b8006a47264df58c819abfbd88888eb0f20d', content=[ResponseOutputText(annotations=[], text='```python\nfrom dotenv import load_dotenv\nimport os\n\nload_dotenv()  # 현재 디렉터리의 .env를 읽음\n\napi_key = os.getenv("OPENAI_API_KEY")\nprint(api_key)\n```\n\n주의점 1. `.env` 파일에는 `OPENAI_API_KEY=your_key_here` 형식으로 넣고, 이 파일이 실행 중인 작업 디렉터리에서', type='output_text', logprobs=[])], role='assistant', status='incomplete', type='message', phase='final_answer')], parallel_tool_calls=True, temperature=1.0, tool_

---
## 9. 조합 비교


In [14]:
comparison_prompt = "LLM API 호출 비용을 줄이는 방법을 5개 알려줘."

settings = [
    {"name": "짧고 안정적", "params": {"max_output_tokens": 120, "reasoning": {"effort": "low"}}},
    {"name": "균형", "params": {"max_output_tokens": 220, "reasoning": {"effort": "medium"}}},
    {"name": "자세하고 신중", "params": {"max_output_tokens": 360, "reasoning": {"effort": "high"}}},
]

for setting in settings:
    generate(setting["name"], comparison_prompt, **setting["params"])



--- 짧고 안정적 ---
LLM API 호출 비용을 줄이는 방법 5가지는 아래와 같습니다.

1. **프롬프트를 짧고 명확하게 만들기**
   - 불필요한 설명, 중복 문장, 긴 예시를 줄이면 입력 토큰이 줄어 비용이 절감됩니다.
   - 시스템 프롬프트도 꼭 필요한 내용만 남기는 게 좋습니다.

2. **출력 길이 제한하기**
   - `max_tokens`

토큰: input=22, output=120, total=142

--- 균형 ---
LLM API 호출 비용을 줄이는 방법 5가지입니다.

1. **입력 토큰을 줄이기**
   - 긴 대화 기록, 불필요한 설명, 중복 문장을 최대한

토큰: input=22, output=220, total=242

--- 자세하고 신중 ---
LLM API 호출 비용을 줄이는 방법 5가지입니다.

1. **프롬프트를 짧고 명확하게 만들기**
   - 시스템/지시문/예시가 길수록 입력 토큰 비용이 늘어납니다.
   - 불필요한 설명, 중복 지시, 긴 예시는 줄이세요.

2. **응답 길이 제한하기**
   - `max_tokens`(또는 `max_output_tokens`)를 적절히 낮게 설정하면 출력 비용을 줄일 수 있습니다.
   - “짧게 답변”, “3줄 이내” 같은 조건도 도움이 됩니다.

3. **캐싱 활용하기**
   - 같은 질문이나 비슷한 요청에 대해 이전 응답을 저장해 재사용합니다.
   - FAQ, 규정 안내, 고정된 요약 작업에서 특히 효과적입니다.

4. **작은 모델을 우선 사용하고, 필요할 때만 큰 모델 쓰기**
   - 간단한 분류, 정리, 초안 생성은 저렴한 모델로 처리하고,
   - 복잡한 추론이나 고난도 작업만 고성능 모델로 보내는

토큰: input=22, output=360, total=382
